In [74]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e7/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e7/train.csv
/kaggle/input/competitions/playground-series-s6e7/test.csv


In [75]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/test.csv")

In [76]:
train.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


In [77]:
mapping = {
    "unhealthy": 0,
    "at-risk": 1,
    "fit": 2
}

train["health_condition"] = train["health_condition"].map(mapping)

In [78]:
train

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,0,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,1,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,0,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,0,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,1,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
690083,690083,1,6.31,69.7,24.11,2157.0,NaN,30.8,3.00,non-veg,high,poor,active,yes,female
690084,690084,1,5.78,54.0,26.36,2858.0,6488.0,52.4,1.46,veg,medium,average,moderate,no,male
690085,690085,2,7.64,85.7,21.91,2195.0,9241.0,41.3,1.57,non-veg,NaN,average,active,no,male
690086,690086,1,6.74,73.0,18.73,2061.0,13366.0,56.6,2.60,balanced,NaN,average,active,yes,male


In [79]:
y = train["health_condition"]

train = train.drop(columns=["health_condition"])

df = pd.concat([train, test], ignore_index=True)

In [80]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 985841 entries, 0 to 985840
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       985841 non-null  int64  
 1   sleep_duration           877271 non-null  float64
 2   heart_rate               974651 non-null  float64
 3   bmi                      965987 non-null  float64
 4   calorie_expenditure      910336 non-null  float64
 5   step_count               965961 non-null  float64
 6   exercise_duration        975982 non-null  float64
 7   water_intake             923731 non-null  float64
 8   diet_type                975982 non-null  object 
 9   stress_level             867540 non-null  object 
 10  sleep_quality            902511 non-null  object 
 11  physical_activity_level  933525 non-null  object 
 12  smoking_alcohol          945010 non-null  object 
 13  gender                   955308 non-null  object 
dtypes: f

In [81]:
stud_ids = test["id"].copy()

In [82]:
stud_ids.head()

0    690088
1    690089
2    690090
3    690091
4    690092
Name: id, dtype: int64

## Filling Nulls

In [83]:
df.isna().sum()

id                              0
sleep_duration             108570
heart_rate                  11190
bmi                         19854
calorie_expenditure         75505
step_count                  19880
exercise_duration            9859
water_intake                62110
diet_type                    9859
stress_level               118301
sleep_quality               83330
physical_activity_level     52316
smoking_alcohol             40831
gender                      30533
dtype: int64

In [84]:
df.drop(columns='id',inplace=True)

In [85]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(include="object").columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    df[col] = df[col].fillna("unknown")

In [86]:
df.isnull().sum()

sleep_duration             0
heart_rate                 0
bmi                        0
calorie_expenditure        0
step_count                 0
exercise_duration          0
water_intake               0
diet_type                  0
stress_level               0
sleep_quality              0
physical_activity_level    0
smoking_alcohol            0
gender                     0
dtype: int64

In [87]:
df.shape

(985841, 13)

In [88]:
train_df = df.iloc[:len(y)].copy()
test_df = df.iloc[len(y):].copy()

In [89]:
train_df

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,unknown,average,sedentary,unknown,male
...,...,...,...,...,...,...,...,...,...,...,...,...,...
690083,6.31,69.7,24.11,2157.0,8856.0,30.8,3.00,non-veg,high,poor,active,yes,female
690084,5.78,54.0,26.36,2858.0,6488.0,52.4,1.46,veg,medium,average,moderate,no,male
690085,7.64,85.7,21.91,2195.0,9241.0,41.3,1.57,non-veg,unknown,average,active,no,male
690086,6.74,73.0,18.73,2061.0,13366.0,56.6,2.60,balanced,unknown,average,active,yes,male


## Feature Engineering

In [90]:
## Numerical columns
df["exercise_water_ratio"] = (
    df["exercise_duration"] / (df["water_intake"] + 1)
)

df["water_per_calorie"] = df["water_intake"] / (df["calorie_expenditure"] + 1)

df["sleep_per_exercise"] = df["sleep_duration"] / (df["exercise_duration"] + 1)

df["bmi_activity"] = (
    df["bmi"] * df["exercise_duration"]
)
df["hydration_score"] = (
    df["water_intake"] * df["sleep_duration"]
)
df["heart_rate_efficiency"] = (
    df["calorie_expenditure"] / (df["heart_rate"] + 1)
)
df["recovery_ratio"] = (
    df["sleep_duration"]
    / (df["calorie_expenditure"] + 1)
)

In [91]:
## Categorical columns
df["diet_activity"] = (
    df["diet_type"].astype(str)
    + "_"
    + df["physical_activity_level"].astype(str)
)

df["stress_sleep"] = (
    df["stress_level"].astype(str)
    + "_"
    + df["sleep_quality"].astype(str)
)

df["gender_activity"] = (
    df["gender"].astype(str)
    + "_"
    + df["physical_activity_level"].astype(str)
)

df["diet_smoking"] = (
    df["diet_type"].astype(str)
    + "_"
    + df["smoking_alcohol"].astype(str)
)

In [92]:
cat_cols = df.select_dtypes(include="object").columns

for col in cat_cols:
    freq = df[col].value_counts()

    df[col + "_freq"] = df[col].map(freq)

In [93]:
df.head()

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,...,diet_type_freq,stress_level_freq,sleep_quality_freq,physical_activity_level_freq,smoking_alcohol_freq,gender_freq,diet_activity_freq,stress_sleep_freq,gender_activity_freq,diet_smoking_freq
0,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,...,330502,253578,305247,315754,319011,310236,106846,78819,100072,107104
1,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,...,321110,239575,305247,316529,319011,304881,101534,73925,99867,104144
2,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,...,330502,253578,303470,301242,319011,340191,99783,82330,104895,107104
3,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,...,330502,253578,305247,301242,311630,310236,99783,78819,95065,104491
4,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,unknown,average,...,330502,118301,305247,315754,40831,340191,106846,36734,108924,13790


## 5-fold CV with Target Encoding

In [94]:
from sklearn.model_selection import StratifiedKFold
from category_encoders import TargetEncoder
from lightgbm import LGBMClassifier
from sklearn.metrics import balanced_accuracy_score
import numpy as np

In [95]:
X = train_df.copy()
y = y.copy()          
X_test = test_df.copy()

In [96]:
cat_cols = X.select_dtypes(include="object").columns.tolist()

print(cat_cols)

['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']


In [97]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [98]:
oof_preds = np.zeros(len(X))

test_preds = np.zeros((len(X_test), 3))

## Ensemble Learning

## 1.LightGBM + Catboost

In [99]:
# model = LGBMClassifier(
#     objective="multiclass",
#     num_class=3,
#     n_estimators=500,
#     learning_rate=0.05,
#     random_state=42
# )

# model = LGBMClassifier(
#     objective="multiclass",
#     num_class=3,
#     learning_rate=0.03,
#     n_estimators=1200,
#     num_leaves=63,
#     max_depth=-1,
#     min_child_samples=30,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     reg_alpha=0.1,
#     reg_lambda=1.0,
#     random_state=42
# )

# model.fit(
#     X_train,
#     y_train
# )

In [100]:
# test_preds += model.predict_proba(X_test_encoded) / skf.n_splits

In [101]:
from catboost import CatBoostClassifier

In [102]:
import lightgbm as lgb

In [ ]:
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):

    print(f"Fold {fold}")

    # Split data
    X_train = X.iloc[train_idx].copy()
    X_val = X.iloc[val_idx].copy()

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    # Target Encoding
    encoder = TargetEncoder(cols=cat_cols)

    X_train[cat_cols] = encoder.fit_transform(
        X_train[cat_cols],
        y_train
    )

    X_val[cat_cols] = encoder.transform(
        X_val[cat_cols]
    )

    X_test_encoded = X_test.copy()

    X_test_encoded[cat_cols] = encoder.transform(
        X_test_encoded[cat_cols]
    )

    # ---------------- LightGBM ----------------
    lgb_model = LGBMClassifier(
        objective="multiclass",
        num_class=3,
        learning_rate=0.03,
        n_estimators=1200,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=30,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42
    )

    lgb_model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="multi_logloss",
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=100,
                verbose=False
            )
        ]
    )

    # ---------------- CatBoost ----------------
    cat_model = CatBoostClassifier(
        loss_function="MultiClass",
        iterations=1200,
        learning_rate=0.03,
        depth=6,
        random_seed=42,
        verbose=0
    )

    cat_model.fit(
        X_train,
        y_train,
        eval_set=(X_val, y_val),
        use_best_model=True
    )

    # Validation predictions
    lgb_val = lgb_model.predict_proba(X_val)
    cat_val = cat_model.predict_proba(X_val)

    val_probs = (lgb_val + cat_val) / 2

    val_pred = np.argmax(val_probs, axis=1)

    oof_preds[val_idx] = val_pred

    # Test predictions
    lgb_test = lgb_model.predict_proba(X_test_encoded)
    cat_test = cat_model.predict_proba(X_test_encoded)

    test_preds += (lgb_test + cat_test) / skf.n_splits

Fold 1
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010764 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1814
[LightGBM] [Info] Number of data points in the train set: 552070, number of used features: 13
[LightGBM] [Info] Start training from score -2.481150
[LightGBM] [Info] Start training from score -0.152364
[LightGBM] [Info] Start training from score -2.852889
Fold 2
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.036595 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1815
[LightGBM] [Info] Number of data points in the train set: 552070, number of used features: 13
[LightGBM] [Info] Start training from score -2.481150
[LightGBM] [Info] Start training from score -0.152364
[LightGBM] [Info] Start training from score -2.852889
Fold 3
[LightGBM] 

In [ ]:
score = balanced_accuracy_score(y, oof_preds)

print(f"CV Balanced Accuracy : {score:.5f}")

## submission

In [ ]:
final_pred = np.argmax(test_preds, axis=1)

In [ ]:
inverse_mapping = {
    0: "unhealthy",
    1: "at-risk",
    2: "fit"
}

submission = pd.DataFrame({
    "id": stud_ids,
    "health_condition": pd.Series(final_pred).map(inverse_mapping)
})


In [ ]:
submission.head()

In [ ]:
submission.to_csv("submission.csv",index=False)